# 5주차 실습 «문제본» — 퍼셉트론

> 이 노트북은 **학생 배포용 문제본**이다. 핵심 로직이 `TODO(n)` 으로 비워져 있고 그 자리에서
> `NotImplementedError` 가 발생한다. 각 TODO 를 채워 넣어 노트북이 처음부터 끝까지 오류 없이
> 실행되도록 만드는 것이 과제다. **위에서부터 순서대로** 실행한다.

### 학습 목표
1. Rosenblatt 퍼셉트론 학습 규칙(오분류 시에만 $\mathbf{w}\leftarrow\mathbf{w}+\eta y_i\mathbf{x}_i$)을 NumPy 로 직접 구현한다.
2. 퍼셉트론 수렴 정리(Novikoff)의 상한 $(R/\gamma)^2$ 을 실험으로 확인하고, 마진이 왜 중요한지 설명할 수 있다.
3. AND·OR 은 되지만 XOR 은 안 되는 이유를 **표현력(선형 분리 가능성)** 의 관점에서 설명할 수 있다.
4. PyTorch 로 단층 퍼셉트론과 2층 MLP 를 만들고, 은닉층이 만든 새 특징 공간에서 XOR 이 풀리는 것을 확인한다.
5. 트랜스포머 블록의 FFN 이 결국 2층 MLP 이며 파라미터의 약 2/3 를 차지함을 직접 세어 확인한다.

### TODO 목록 (총 15개)
| # | 파트 | 무엇을 구현하는가 |
|---|---|---|
| 1 | 1 · NumPy | 퍼셉트론 학습 규칙: 오분류 판정과 가중치 갱신 |
| 2 | 1 · NumPy | 학습된 $\mathbf{w}$ 로부터 결정 경계 직선 좌표 계산 |
| 3 | 1 · NumPy | 데이터 반지름 $R$ 과 Novikoff 상한 $(R/\gamma)^2$ 계산 |
| 4 | 2 · 논리 게이트 | 진리표를 $\{-1,+1\}$ 레이블로 바꾸고 게이트별 퍼셉트론 학습 |
| 5 | 2 · 논리 게이트 | 학습된 $\mathbf{w}$ 로 게이트 출력 예측(계단 함수) |
| 6 | 3 · sklearn | 표준화 + `Perceptron` 파이프라인 구성·학습·예측 |
| 7 | 4 · PyTorch | 학습 루프 한 스텝: 순전파 → BCE 손실 → zero_grad → backward → step |
| 8 | 4 · PyTorch | XOR 을 푸는 2층 MLP (2→2→1, tanh) 정의 |
| 9 | 4 · PyTorch | 격자 전체에 대한 예측 확률(결정 경계용) 계산 |
| 10 | 4 · PyTorch | 은닉층 출력 $\mathbf{h}=\tanh(W_1\mathbf{x}+\mathbf{b}_1)$ 계산 |
| 11 | 4-b · PyTorch | digits 미니배치 학습 루프 (교차 엔트로피) |
| 12 | 4-b · PyTorch | 테스트 정확도 계산 |
| 13 | 4-b · PyTorch | 2층 MLP (64→128→10, GELU+Dropout) 정의·학습 |
| 14 | 5 · 트랜스포머 | `TransformerEncoderLayer` 의 FFN 파라미터 비율 계산 |
| 15 | 5 · 트랜스포머 | FFN 첫 선형층 + ReLU 통과시켜 뉴런 발화 확인 |

### 제출물
- 모든 TODO 를 채우고 **처음부터 끝까지 실행한 결과(그림 포함)가 저장된** `5주차_perceptron_pytorch.ipynb`
- 마지막 마크다운 셀의 «과제(선택)» 중 최소 1개에 대한 짧은 답변(노트북 맨 끝에 마크다운 셀로 추가)

### 예상 소요 시간
- TODO 구현 60~90분 + 전체 실행 1분 내외(CPU로 충분, GPU 불필요)

### 자가 점검
- `Kernel → Restart & Run All` 로 `NotImplementedError` 없이 끝까지 실행되면 1차 통과.
- 기대 결과: 마진 0.3 에서 4회 갱신 · AND/OR 정확도 1.00, XOR 0.50 · sklearn digits 0.94 내외 ·
  XOR MLP 정확도 1.00 · digits 선형 0.96 / MLP 0.97 내외 · FFN 파라미터 비율 약 66%.

# 5주차 실습 — 퍼셉트론: 학습 규칙, 논리 게이트, 그리고 MLP로의 확장 (2026 개정판)

**기계학습특론** · 최승호 (jcn99250@naver.com)

이번 실습은 다섯 부분으로 구성됩니다.

| 파트 | 내용 | 라이브러리 |
|---|---|---|
| 1 | Rosenblatt 퍼셉트론 학습 규칙 직접 구현, 수렴 정리 실험 (마진 vs. 갱신 횟수) | NumPy |
| 2 | 논리 게이트(AND/OR/XOR)와 선형 분리 가능성 | NumPy |
| 3 | scikit-learn `Perceptron`으로 필기체 숫자 분류 (기존 실습 갱신) | scikit-learn 1.8 |
| 4 | PyTorch로 퍼셉트론(=로지스틱 회귀)과 2층 MLP 구현, XOR 해결 | PyTorch 2.x |
| 5 | 트랜스포머 블록 안의 "퍼셉트론" 찾기 — FFN은 2층 MLP다 | PyTorch |

> 실행 환경: Python ≥ 3.10, `pip install numpy scikit-learn matplotlib torch`. GPU 불필요.


In [ ]:
import numpy as np, matplotlib.pyplot as plt, torch, sklearn
print('numpy', np.__version__, '| torch', torch.__version__, '| sklearn', sklearn.__version__)
SEED = 0
np.random.seed(SEED); torch.manual_seed(SEED)

# 한글 그래프 폰트 설정 (Windows: Malgun Gothic, macOS: AppleGothic, Colab: 아래 주석 참고)
import matplotlib
from matplotlib import font_manager
_avail = {f.name for f in font_manager.fontManager.ttflist}
for _f in ['Malgun Gothic', 'NanumGothic', 'AppleGothic', 'Noto Sans CJK KR', 'Noto Sans CJK SC', 'Noto Sans CJK JP']:
    if _f in _avail:
        matplotlib.rcParams['font.family'] = _f; break
matplotlib.rcParams['axes.unicode_minus'] = False
# Colab: !apt-get -qq install fonts-nanum && 런타임 재시작 후 위 셀 재실행
print('그래프 폰트:', matplotlib.rcParams['font.family'])

## 파트 1. Rosenblatt 퍼셉트론 학습 규칙 (1958)

퍼셉트론은 특징 벡터 $\mathbf{x}\in\mathbb{R}^d$ (바이어스 성분 $x_0=1$ 포함)에 대해

$$\hat y = \operatorname{sgn}(\mathbf{w}^\top \mathbf{x}),\qquad y\in\{-1,+1\}$$

로 예측하고, **오분류된 샘플에 대해서만** 다음 규칙으로 가중치를 갱신합니다.

$$\mathbf{w} \leftarrow \mathbf{w} + \eta\, y_i\, \mathbf{x}_i \quad (\text{if } y_i \mathbf{w}^\top\mathbf{x}_i \le 0)$$

이 규칙은 손실 $L(\mathbf{w}) = \sum_{i\in\mathcal{M}} -y_i\mathbf{w}^\top\mathbf{x}_i$ (오분류 집합 $\mathcal{M}$)에 대한 확률적 경사 하강과 동일합니다.

**퍼셉트론 수렴 정리 (Novikoff, 1962).** 데이터가 마진 $\gamma>0$으로 선형 분리 가능하고 $\|\mathbf{x}_i\|\le R$이면, 갱신 횟수는 $\le (R/\gamma)^2$ 입니다. 아래에서 마진을 바꿔 가며 이를 실험으로 확인합니다.


In [ ]:
def perceptron_train(X, y, lr=1.0, max_epochs=1000):
    """Rosenblatt 학습 규칙. X: (N,d) (바이어스 열 포함), y: {-1,+1}. 반환: w, 갱신 횟수, 에폭별 오류 수"""
    w = np.zeros(X.shape[1]); n_updates = 0; history = []
    for epoch in range(max_epochs):
        errors = 0
        for xi, yi in zip(X, y):
            # TODO(1): 오분류(또는 경계 위, 즉 y_i * (w·x_i) <= 0)인 샘플에 대해서만
            #          w <- w + lr * y_i * x_i 로 갱신하고 n_updates 와 errors 를 1씩 늘린다.
            #   힌트: 넘파이 내적은 w @ xi. 올바르게 분류된 샘플은 건드리지 않는다.
            raise NotImplementedError("TODO(1)")
        history.append(errors)
        if errors == 0: break               # 모든 샘플이 올바르게 분류되면 종료
    return w, n_updates, history

def make_separable(n=200, margin=0.5, d=2, seed=0):
    """진짜 분리 초평면 w*로부터 마진 이상 떨어진 점만 남겨 선형 분리 가능 데이터 생성"""
    rng = np.random.default_rng(seed)
    w_star = rng.normal(size=d); w_star /= np.linalg.norm(w_star)
    X = rng.uniform(-1, 1, size=(4*n, d))
    s = X @ w_star
    keep = np.abs(s) >= margin
    X, s = X[keep][:n], s[keep][:n]
    y = np.where(s > 0, 1, -1)
    Xb = np.hstack([np.ones((len(X),1)), X])   # x0 = 1 (바이어스)
    return Xb, y, w_star

Xb, y, w_star = make_separable(margin=0.3)
w, n_upd, hist = perceptron_train(Xb, y)
print(f'갱신 횟수 = {n_upd}, 에폭 수 = {len(hist)}, 최종 훈련 오류 = {hist[-1]}')

fig, ax = plt.subplots(1, 2, figsize=(10,4))
ax[0].scatter(Xb[:,1], Xb[:,2], c=y, cmap='coolwarm', s=15)
xs = np.linspace(-1,1,2)
# TODO(2): 학습된 w 로 결정 경계를 그릴 좌표를 계산하라 (변수 이름은 boundary_x2).
#          경계는 w0 + w1*x1 + w2*x2 = 0 이므로 이를 x2 에 대해 풀면 된다.
#   힌트: xs 배열의 각 x1 값에 대응하는 x2 값을 한 번에 계산한다.
raise NotImplementedError("TODO(2)")
ax[0].plot(xs, boundary_x2, 'k-', label='학습된 경계')
ax[0].set_title('퍼셉트론 결정 경계'); ax[0].set_xlim(-1,1); ax[0].set_ylim(-1,1); ax[0].legend()
ax[1].plot(hist, 'o-'); ax[1].set_xlabel('epoch'); ax[1].set_ylabel('오분류 수'); ax[1].set_title('학습 곡선')
plt.tight_layout(); plt.show()

### 실험: 마진 $\gamma$가 작아질수록 갱신 횟수가 $(R/\gamma)^2$ 스케일로 증가하는가?

In [ ]:
margins = [0.4, 0.2, 0.1, 0.05, 0.025]
rows = []
for g in margins:
    ups = []
    for seed in range(10):
        Xb, y, _ = make_separable(margin=g, seed=seed)
        _, n_upd, _ = perceptron_train(Xb, y, max_epochs=5000)
        ups.append(n_upd)
    # TODO(3): 데이터 반지름 R(= ||x||의 상한)과 Novikoff 상한 (R/γ)^2 을 계산해
    #          rows 에 (γ, 평균 갱신 횟수, 상한) 튜플을 추가하라.
    #   힌트: 바이어스 x0=1 이고 x1,x2 ∈ [-1,1] 이므로 ||x|| ≤ sqrt(1+1+1) 이다.
    raise NotImplementedError("TODO(3)")
print(f"{'margin':>8} {'평균 갱신횟수':>12} {'(R/γ)^2 상한':>14}")
for g, m, b in rows: print(f'{g:8.3f} {m:12.1f} {b:14.1f}')

plt.loglog([r[0] for r in rows], [r[1] for r in rows], 'o-', label='실측 평균 갱신 횟수')
plt.loglog([r[0] for r in rows], [r[2] for r in rows], 's--', label='Novikoff 상한 $(R/\\gamma)^2$')
plt.gca().invert_xaxis(); plt.xlabel('margin γ'); plt.ylabel('updates'); plt.legend(); plt.grid(True, which='both', alpha=.3)
plt.show()

**해석.** 실측 갱신 횟수는 항상 상한 아래에 있으며, 마진이 절반이 되면 대략 4배 가까이 증가하는 $1/\gamma^2$ 경향을 보입니다. 상한은 *최악의 샘플 순서*에 대한 것이므로 실측치는 보통 훨씬 작습니다.

> **심화 (대학원):** 퍼셉트론은 분리 초평면을 *아무거나* 찾지만, 로지스틱 손실로 경사 하강을 무한히 돌리면 $\mathbf{w}/\|\mathbf{w}\|$ 가 **최대 마진(hard-margin SVM) 방향으로 수렴**합니다 (Soudry et al., JMLR 2018, "implicit bias"). 최근 연구는 다중 클래스(NeurIPS 2024)와 Muon·스펙트럴 디센트(2025, 스펙트럴 노름 기준 최대 마진)로 확장되었습니다. 강의 슬라이드 참고.


## 파트 2. 논리 게이트와 선형 분리 가능성

입력 $(x_1,x_2)\in\{0,1\}^2$ 에 대해 AND, OR, XOR 을 퍼셉트론으로 학습시켜 봅니다. 활성화 함수는 계단 함수이며 출력은 $\{0,1\}$ 로 표현합니다.


In [ ]:
X_gate = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
Xb_gate = np.hstack([np.ones((4,1)), X_gate])
gates = {'AND': np.array([0,0,0,1]), 'OR': np.array([0,1,1,1]), 'XOR': np.array([0,1,1,0])}

fig, axes = plt.subplots(1, 3, figsize=(12,3.8))
for ax, (name, t) in zip(axes, gates.items()):
    # TODO(4): 진리표 t (0/1) 를 퍼셉트론 레이블 y_pm ∈ {-1,+1} 로 바꾸고,
    #          perceptron_train(Xb_gate, y_pm, max_epochs=100) 으로 학습해 w, n_upd, hist 를 얻어라.
    #   힌트: np.where 로 0 → -1, 1 → +1.
    raise NotImplementedError("TODO(4)")
    # TODO(5): 학습된 w 로 네 입력의 게이트 출력을 0/1 로 예측해 pred 에 담아라.
    #   힌트: 계단 함수이므로 (Xb_gate @ w > 0) 을 정수형으로 바꾼다.
    raise NotImplementedError("TODO(5)")
    print(f'{name}: w = {np.round(w,2)}, 갱신 {n_upd}회, 훈련 정확도 = {np.mean(pred==t):.2f}')
    ax.scatter(X_gate[:,0], X_gate[:,1], c=t, cmap='coolwarm', s=120, edgecolor='k')
    if abs(w[2]) > 1e-9:
        xs = np.linspace(-.5,1.5,2); ax.plot(xs, -(w[0]+w[1]*xs)/w[2], 'k--')
    ax.set_title(f'{name} (acc={np.mean(pred==t):.2f})'); ax.set_xlim(-.5,1.5); ax.set_ylim(-.5,1.5)
plt.tight_layout(); plt.show()

**결과.** AND/OR 은 몇 번의 갱신으로 수렴하지만 XOR 은 `max_epochs` 를 다 써도 정확도가 0.5~0.75 를 넘지 못합니다. 이는 알고리즘의 문제가 아니라 **표현력의 한계**입니다 (Minsky & Papert, 1969) — 단일 초평면으로는 XOR 의 두 클래스를 나눌 수 없습니다. 해결책은 파트 4에서 다룹니다.


## 파트 3. scikit-learn `Perceptron` — 필기체 숫자 분류 (기존 실습 갱신)

sklearn 의 `Perceptron` 은 다중 클래스에 대해 자동으로 one-vs-rest 를 수행합니다. 기존 실습 코드를 최신 API 관례(`ConfusionMatrixDisplay`, `StandardScaler` 파이프라인)로 정리했습니다.


In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Perceptron
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score

X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1, stratify=y)

# TODO(6): 표준화(StandardScaler) → Perceptron(tol=1e-3, random_state=0) 파이프라인 clf 를 만들고
#          훈련 데이터로 학습한 뒤, 테스트셋 예측을 y_pred 에 담아라.
#   힌트: make_pipeline 을 쓰면 clf[-1] 로 마지막 단계(퍼셉트론)에 접근할 수 있다.
#         다중 클래스는 sklearn 이 자동으로 one-vs-rest 로 처리한다.
raise NotImplementedError("TODO(6)")
print('테스트 정확도 =', round(accuracy_score(y_test, y_pred), 4))
print(classification_report(y_test, y_pred, digits=3))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred); plt.show()

# 가중치 벡터를 8x8 이미지로 시각화: 각 클래스 퍼셉트론이 "어떤 픽셀"에 반응하는지
W = clf[-1].coef_
fig, axes = plt.subplots(2,5, figsize=(10,4))
for k, ax in enumerate(axes.ravel()):
    ax.imshow(W[k].reshape(8,8), cmap='RdBu'); ax.set_title(f'digit {k}'); ax.axis('off')
plt.suptitle('one-vs-rest 퍼셉트론 가중치 (빨강: 양의 증거, 파랑: 음의 증거)'); plt.show()

## 파트 4. PyTorch — 퍼셉트론(로지스틱 회귀)과 2층 MLP

* **활성화를 시그모이드로 바꾼 퍼셉트론 = 로지스틱 회귀** 입니다. `nn.Linear(2,1)` + `BCEWithLogitsLoss` 로 구현하면 손실이 볼록(convex)이라 경사 하강이 전역 최적에 도달합니다.
* **은닉층 하나를 추가한 2층 MLP** 는 XOR 을 풀 수 있습니다 (은닉 뉴런 2개면 충분).
* 최신 관례: `torch.optim.AdamW`, `torch.nn.functional`, 명시적 시드, 배치 전체를 텐서로 처리.


In [ ]:
import torch.nn as nn, torch.nn.functional as F

Xt = torch.tensor(X_gate, dtype=torch.float32)
yt_xor = torch.tensor(gates['XOR'], dtype=torch.float32).unsqueeze(1)

def train_torch(model, X, y, epochs=2000, lr=0.1, wd=0.0, log_every=500):
    """AdamW 로 이진 분류 모델을 학습한다. 반환: 학습이 끝난 model"""
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    for ep in range(1, epochs+1):
        # TODO(7): 학습 한 스텝을 구현하라 — 순전파로 logits 를 얻고,
        #          BCEWithLogits 손실 loss 를 계산한 뒤, zero_grad → backward → step 순서로 갱신한다.
        #   힌트: F.binary_cross_entropy_with_logits(logits, y)
        raise NotImplementedError("TODO(7)")
        if ep % log_every == 0 or ep == 1:
            acc = ((logits > 0).float() == y).float().mean().item()
            print(f'  epoch {ep:5d}  loss={loss.item():.4f}  acc={acc:.2f}')
    return model

torch.manual_seed(0)
print('[1] 단층 퍼셉트론 (nn.Linear 2→1) on XOR')
single = nn.Linear(2, 1)
train_torch(single, Xt, yt_xor)

torch.manual_seed(0)
print('\n[2] 2층 MLP (2→2→1, tanh) on XOR')
# TODO(8): 은닉 뉴런 2개와 tanh 활성화를 쓰는 2층 MLP (2→2→1) 를 nn.Sequential 로 정의해 mlp 에 담아라.
#   힌트: Linear → Tanh → Linear. 은닉 2개면 XOR 을 풀기에 충분하다.
raise NotImplementedError("TODO(8)")
train_torch(mlp, Xt, yt_xor)

### 결정 경계와 은닉 특징 공간 시각화 — TODO(9), TODO(10)

위에서 학습한 두 모델의 결정 경계를 그리고, 은닉층이 XOR 을 선형 분리 가능하게 재배치하는지 확인한다.

In [ ]:
def plot_boundary(model, title, ax):
    xx, yy = np.meshgrid(np.linspace(-.5,1.5,200), np.linspace(-.5,1.5,200))
    grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
    # TODO(9): 그리드 전체에 대한 모델의 예측 확률 zz 를 계산하라 (xx 와 같은 모양의 넘파이 배열).
    #   힌트: 기울기가 필요 없으니 with torch.no_grad(): 안에서 torch.sigmoid(model(grid)) 를 구하고
    #         .numpy().reshape(xx.shape) 로 모양을 맞춘다.
    raise NotImplementedError("TODO(9)")
    ax.contourf(xx, yy, zz, levels=20, cmap='coolwarm', alpha=.8)
    ax.contour(xx, yy, zz, levels=[0.5], colors='k')
    ax.scatter(X_gate[:,0], X_gate[:,1], c=gates['XOR'], cmap='coolwarm', s=120, edgecolor='k')
    ax.set_title(title)

fig, axes = plt.subplots(1,2, figsize=(9,4))
plot_boundary(single, '단층 퍼셉트론: XOR 실패', axes[0])
plot_boundary(mlp, '2층 MLP: XOR 해결', axes[1])
plt.tight_layout(); plt.show()

# 은닉층이 만든 새로운 특징 공간에서는 XOR가 선형 분리 가능해진다
# TODO(10): 네 입력에 대한 은닉층 출력 h(x) = tanh(W1 x + b1) 을 계산해 넘파이 배열 H 에 담아라.
#   힌트: mlp[0] 이 첫 번째 Linear 층이다. with torch.no_grad(): 안에서 계산한다.
raise NotImplementedError("TODO(10)")
print('은닉층 출력 h(x):'); print(np.round(H, 2))

**핵심 메시지.** 은닉층은 입력을 *새로운 좌표계* $\mathbf{h}=\phi(W_1\mathbf{x}+\mathbf{b}_1)$ 로 옮기고, 출력층은 그 공간에서 **다시 퍼셉트론**을 적용합니다. 위 `H` 출력에서 XOR 의 두 클래스가 선형 분리 가능하게 재배치된 것을 확인하세요. 이 "특징 학습 + 선형 분류기" 구조가 CNN·트랜스포머까지 이어지는 딥러닝의 기본 골격입니다.


### 4-b. PyTorch MLP 로 필기체 숫자 분류 — 파트 3의 선형 퍼셉트론과 비교

In [ ]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler().fit(X_train)
Xtr = torch.tensor(sc.transform(X_train), dtype=torch.float32); ytr = torch.tensor(y_train)
Xte = torch.tensor(sc.transform(X_test),  dtype=torch.float32); yte = torch.tensor(y_test)

def fit_classifier(model, epochs=60, lr=1e-3, bs=64):
    """미니배치 SGD(AdamW)로 다중 클래스 분류기를 학습하고 테스트 정확도(float)를 반환한다."""
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    n = len(Xtr)
    for ep in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, bs):
            idx = perm[i:i+bs]
            # TODO(11): 미니배치 한 스텝을 구현하라 — 순전파 → 교차 엔트로피 손실 →
            #           zero_grad → backward → step.
            #   힌트: F.cross_entropy(model(Xtr[idx]), ytr[idx])
            raise NotImplementedError("TODO(11)")
    with torch.no_grad():
        # TODO(12): 테스트셋 정확도를 계산해 파이썬 float 로 반환하라.
        #   힌트: model(Xte).argmax(1) 과 yte 를 비교한 뒤 평균을 내고 .item() 을 쓴다.
        raise NotImplementedError("TODO(12)")

torch.manual_seed(0)
acc_lin = fit_classifier(nn.Linear(64, 10))                                   # 다중클래스 퍼셉트론(softmax 회귀)
torch.manual_seed(0)
# TODO(13): 은닉 128 + GELU + Dropout(0.1) 을 쓰는 2층 MLP (64→128→10) 를 정의하고
#           fit_classifier 로 학습·평가해 정확도를 acc_mlp 에 담아라.
#   힌트: nn.Sequential(nn.Linear(...), nn.GELU(), nn.Dropout(0.1), nn.Linear(...))
raise NotImplementedError("TODO(13)")
print(f'선형(softmax 퍼셉트론) 테스트 정확도 = {acc_lin:.4f}')
print(f'2층 MLP(GELU)      테스트 정확도 = {acc_mlp:.4f}')

## 파트 5. 트랜스포머 블록 안의 퍼셉트론

LLM 을 구성하는 트랜스포머 블록은 **어텐션**과 **FFN(feed-forward network)** 의 반복입니다. FFN 은 정확히 우리가 방금 만든 2층 MLP 이며 (`Linear → 활성화 → Linear`), GPT 계열 모델 파라미터의 약 2/3 가 여기에 있습니다. 아래에서 PyTorch 의 `TransformerEncoderLayer` 를 열어 확인합니다.


In [ ]:
layer = nn.TransformerEncoderLayer(d_model=64, nhead=4, dim_feedforward=256, batch_first=True)
print(layer)
# TODO(14): FFN(= linear1 + linear2) 의 파라미터 수 n_ffn 과 이 층 전체 파라미터 수 n_all 을 구하라.
#   힌트: layer.named_parameters() 에서 이름에 'linear' 가 들어간 것만 골라 p.numel() 을 더한다.
raise NotImplementedError("TODO(14)")
print(f'\nFFN(2층 MLP) 파라미터 비율 = {n_ffn/n_all:.1%}')

# 하나의 FFN 뉴런 = 하나의 퍼셉트론: w1·x + b1 → 활성화 → 출력 가중치 w2
x = torch.randn(1, 5, 64)                     # (batch, tokens, d_model)
# TODO(15): FFN 의 첫 번째 선형층에 x 를 통과시키고 ReLU 를 적용한 h 를 구하라.
#           (256개의 "퍼셉트론"이 각 토큰에 대해 발화 여부를 결정한다)
#   힌트: layer.linear1 과 F.relu 를 쓴다.
raise NotImplementedError("TODO(15)")
print('활성화된 FFN 뉴런 비율(토큰별):', (h > 0).float().mean(dim=-1).squeeze().numpy().round(2))

**정리.**
1. 퍼셉트론 = 가중합 + 비선형 활성화. 학습 규칙은 오분류 샘플에 대한 SGD 이고, 분리 가능하면 $(R/\gamma)^2$ 회 이내에 수렴한다.
2. 단일 퍼셉트론은 선형 분리 가능한 문제(AND, OR)만 풀 수 있고 XOR 은 못 푼다.
3. 은닉층을 쌓으면(MLP) 표현력이 보편 근사(universal approximation)로 확장되고, 학습은 역전파로 한다.
4. 트랜스포머·LLM 의 FFN 은 2층 MLP 이며, 어텐션도 "학습된 가중합"이라는 점에서 퍼셉트론의 직계 후손이다.

**과제(선택).** (a) 파트 1에서 학습률 $\eta$ 를 바꿔도 갱신 횟수가 변하지 않는 이유를 수식으로 설명하시오. (b) 파트 4의 MLP 에 은닉 뉴런을 1개만 두면 XOR 을 풀 수 있는지 실험하고 이유를 설명하시오. (c) 파트 4-b 에서 은닉층 폭을 32, 128, 512 로 바꿔 정확도와 파라미터 수를 표로 정리하시오.
